## Phase 1: Problem, Metric & File Structure Analysis 1-2

In [8]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

# ตั้งค่า Seed กลางเพื่อให้ผลลัพธ์คงที่ทุกครั้งที่รัน
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ตรวจสอบ Path อัตโนมัติตามโครงสร้าง VS Code (CPE342-Hackathon/task2/)
if Path('train.csv').exists():
    DATA_DIR = Path('.')
    SUB_PATH = Path('../submission_final.csv')
else:
    DATA_DIR = Path('task2')
    SUB_PATH = Path('submission_final.csv')

assert (DATA_DIR / 'train.csv').exists(), "ไม่พบ train.csv กรุณาเช็กโฟลเดอร์ใน VS Code"
assert SUB_PATH.exists(), "ไม่พบ submission_final.csv กรุณาเช็กตำแหน่งไฟล์"

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sub = pd.read_csv(SUB_PATH)

print(f"โหลดข้อมูลสำเร็จจาก: {DATA_DIR.resolve()}")
print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")
print(f"Sub shape:   {sub.shape} (ไฟล์อยู่ที่: {SUB_PATH.resolve()})")

โหลดข้อมูลสำเร็จจาก: C:\Users\panuw\CPE342-Hackathon\task2
Train shape: (104000, 46)
Test shape:  (25889, 45)
Sub shape:   (25889, 6) (ไฟล์อยู่ที่: C:\Users\panuw\CPE342-Hackathon\submission_final.csv)


In [9]:
TARGET_COL = 'segment'
ID_COL = 'player_id'

# 1. ตรวจสอบสัดส่วน Target (4 Classes)
print("=== Target 'segment' Distribution ===")
target_summary = pd.DataFrame({
    'count': train[TARGET_COL].value_counts().sort_index(),
    'ratio (%)': (train[TARGET_COL].value_counts(normalize=True).sort_index() * 100).round(2)
})
print(target_summary)

# 2. แยกประเภทคอลัมน์ Feature (44 ฟีเจอร์)
feature_cols = [c for c in test.columns if c != ID_COL]
num_cols = train[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
cat_cols = train[feature_cols].select_dtypes(include=['object']).columns.tolist()

print(f"\nจำนวน Features ทั้งหมด: {len(feature_cols)} (ตัวเลข {len(num_cols)} คอลัมน์, ข้อความ {len(cat_cols)} คอลัมน์)")

# 3. ตรวจสอบโครงสร้างไฟล์ submission_final.csv เพื่อป้องกันการเขียนทับผิดคอลัมน์
print("\n=== Inspection: submission_final.csv ===")
print("Columns:", list(sub.columns))
print("Null counts per column:\n", sub.isnull().sum())
print("\nตัวอย่าง 6 แถวแรกของ submission_final.csv:")
print(sub.head(6))

=== Target 'segment' Distribution ===
         count  ratio (%)
segment                  
0        40957      39.38
1        26032      25.03
2        21029      20.22
3        15982      15.37

จำนวน Features ทั้งหมด: 44 (ตัวเลข 34 คอลัมน์, ข้อความ 10 คอลัมน์)

=== Inspection: submission_final.csv ===
Columns: ['id', 'task1', 'task2', 'task3', 'task4', 'task5']
Null counts per column:
 id       0
task1    0
task2    0
task3    0
task4    0
task5    0
dtype: int64

ตัวอย่าง 6 แถวแรกของ submission_final.csv:
         id  task1  task2    task3  task4  task5
0  ANS00001      1      3   370.75      2      1
1  ANS00002      1      2  5036.10      2      0
2  ANS00003      1      2     0.00      3      1
3  ANS00004      1      1  1076.91      0      0
4  ANS00005      0      0  1819.13      3      0
5  ANS00006      1      0   417.28      2      0


## Phase 2: EDA & Data Cleaning 3-5

In [10]:
# 1. ตรวจสอบสัดส่วนค่าว่าง (%) ระหว่าง Train และ Test
missing_summary = pd.DataFrame({
    'train_null_%': (train[feature_cols].isnull().mean() * 100).round(2),
    'test_null_%':  (test[feature_cols].isnull().mean() * 100).round(2),
}).sort_values('train_null_%', ascending=False)

print("=== Top 10 คอลัมน์ที่มีค่าว่าง (Missing Values %) ===")
print(missing_summary.head(10))

# 2. ตรวจสอบ Train-Test Drift (Standardized Mean Difference)
drift = ((train[num_cols].mean() - test[num_cols].mean()).abs() / (train[num_cols].std() + 1e-9)).sort_values(ascending=False)
print(f"\nค่า Train-Test Drift สูงสุด: {drift.max():.4f} SD (หากต่ำกว่า 0.05 แสดงว่า Train และ Test แจกแจงเหมือนกันมาก)")

# 3. ตรวจสอบค่าผิดปกติ (Min / Max ของฟีเจอร์ตัวเลข)
print("\n=== ตัวอย่างช่วงค่า Min - Max ของฟีเจอร์สำคัญ ===")
key_num = ['play_frequency', 'total_playtime_hours', 'total_spending_thb', 'avg_monthly_spending', 'vip_tier', 'win_rate_ranked']
print(train[key_num].agg(['min', 'median', 'mean', 'max']).T.round(2))

=== Top 10 คอลัมน์ที่มีค่าว่าง (Missing Values %) ===
                           train_null_%  test_null_%
payment_method                    30.82        30.40
region                            26.18        26.09
engagement_level                   7.85         7.74
team_play_percentage               7.83         7.57
random_metric_2                    7.81         7.71
platform                           7.81         7.55
ranked_participation_rate          7.78         7.88
avg_monthly_spending               7.77         7.62
tournament_entries                 7.74         7.74
chat_activity_score                7.73         7.61

ค่า Train-Test Drift สูงสุด: 0.0132 SD (หากต่ำกว่า 0.05 แสดงว่า Train และ Test แจกแจงเหมือนกันมาก)

=== ตัวอย่างช่วงค่า Min - Max ของฟีเจอร์สำคัญ ===
                         min    median      mean        max
play_frequency          1.00     11.41     12.14      45.10
total_playtime_hours  100.00   1643.10   1704.79    6019.91
total_spending_thb    501.19  31

In [11]:
# 1. วัดระยะห่างค่าเฉลี่ยระหว่าง 4 คลาสของฟีเจอร์ตัวเลข (Normalized Range)
class_means = train.groupby(TARGET_COL)[num_cols].mean()
sep_score = ((class_means.max() - class_means.min()) / (train[num_cols].std() + 1e-9)).sort_values(ascending=False)

print("=== Top 12 Numeric Features ที่แยกคลาสได้ดีที่สุด ===")
print(sep_score.head(12).round(4))

print("\n=== Bottom 6 Numeric Features (แยกคลาสแทบไม่ได้ / Noise) ===")
print(sep_score.tail(6).round(4))

# 2. ดูสัดส่วนคลาสของ Categorical Features หลัก
for col in ['engagement_level', 'loyalty_tier']:
    print(f"\n=== สัดส่วน Segment แยกตาม {col} ===")
    print(pd.crosstab(train[col].fillna('MISSING'), train[TARGET_COL], normalize='index').round(3))

=== Top 12 Numeric Features ที่แยกคลาสได้ดีที่สุด ===
vip_tier                 1.0911
spending_frequency       0.9442
avg_monthly_spending     0.9273
rare_items_count         0.8148
total_spending_thb       0.7568
days_since_last_login    0.7373
total_playtime_hours     0.5965
friend_count             0.5545
avg_session_duration     0.5368
friend_invites_sent      0.5325
gifts_sent_received      0.5224
tournament_entries       0.4999
dtype: float64

=== Bottom 6 Numeric Features (แยกคลาสแทบไม่ได้ / Noise) ===
random_metric_3        0.0136
avg_match_length       0.0096
account_age_days       0.0079
preferred_game_mode    0.0075
random_metric_1        0.0065
random_metric_2        0.0048
dtype: float64

=== สัดส่วน Segment แยกตาม engagement_level ===
segment               0      1      2      3
engagement_level                            
High              0.184  0.448  0.095  0.273
Low               0.575  0.080  0.296  0.049
MISSING           0.393  0.251  0.203  0.152
Mid             

In [12]:
def clean_data(df_train, df_test):
    tr = df_train.copy()
    te = df_test.copy()
    
    # 1. ลบ player_id และฟีเจอร์ขยะ (Random Noise) ที่ระบุใน datacard.txt
    drop_cols = ['player_id', 'random_metric_1', 'random_metric_2', 'random_metric_3']
    tr = tr.drop(columns=[c for c in drop_cols if c in tr.columns])
    te = te.drop(columns=[c for c in drop_cols if c in te.columns])
    
    # 2. แปลงคอลัมน์ระดับชั้น (Ordinal) เป็นตัวเลขตามลำดับศักดิ์ (คง NaN ไว้ให้โมเดลเรียนรู้เอง)
    engagement_map = {'Low': 1.0, 'Mid': 2.0, 'High': 3.0, 'Veteran': 4.0}
    loyalty_map = {'Bronze': 1.0, 'Silver': 2.0, 'Gold': 3.0, 'Platinum': 4.0, 'Diamond': 5.0}
    skill_map = {'Bronze': 1.0, 'Silver': 2.0, 'Gold': 3.0, 'Platinum': 4.0, 'Diamond': 5.0}
    
    for d in [tr, te]:
        d['engagement_level_ord'] = d['engagement_level'].map(engagement_map)
        d['loyalty_tier_ord'] = d['loyalty_tier'].map(loyalty_map)
        d['skill_tier_ord'] = d['skill_tier'].map(skill_map)
        
    # 3. แปลงคอลัมน์ข้อความ (object) ทั้งหมดเป็น category (แทน NaN ด้วย 'MISSING')
    cat_columns = tr.select_dtypes(include=['object']).columns.tolist()
    for c in cat_columns:
        tr[c] = tr[c].fillna('MISSING').astype('category')
        te[c] = te[c].fillna('MISSING').astype('category')
        
    return tr, te

train_clean, test_clean = clean_data(train, test)
print(f"ทำความสะอาดข้อมูลเสร็จสิ้น -> train_clean: {train_clean.shape}, test_clean: {test_clean.shape}")

ทำความสะอาดข้อมูลเสร็จสิ้น -> train_clean: (104000, 45), test_clean: (25889, 44)


## Phase 3: Solid Baseline & Safe Submission Pipeline 6-8

In [13]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report

# ตรวจสอบว่ามี LightGBM ในเครื่องหรือไม่ ถ้าไม่มีให้ใช้ HistGradientBoostingClassifier แทนอัตโนมัติ
try:
    import lightgbm as lgb
    USE_LGB = True
    print("Model Engine: LightGBM")
except ImportError:
    from sklearn.ensemble import HistGradientBoostingClassifier
    USE_LGB = False
    print("Model Engine: HistGradientBoostingClassifier (sklearn)")

# แยก Features (X) และ Target (y)
X = train_clean.drop(columns=[TARGET_COL])
y = train_clean[TARGET_COL].values
X_test = test_clean.copy()

# ตั้งค่า 5-Fold Stratified CV เพื่อรักษาสัดส่วนทั้ง 4 คลาสให้เท่ากันทุก Fold
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)

# เตรียม Array เก็บความน่าจะเป็น (Probabilities) สำหรับทำ Threshold Tuning และ Ensembling ใน Phase ถัดไป
oof_probs = np.zeros((len(X), 4))
test_probs = np.zeros((len(X_test), 4))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_tr, y_tr = X.iloc[train_idx], y[train_idx]
    X_val, y_val = X.iloc[val_idx], y[val_idx]
    
    if USE_LGB:
        model = lgb.LGBMClassifier(
            objective='multiclass',
            num_class=4,
            n_estimators=1000,
            learning_rate=0.05,
            num_leaves=31,
            subsample_for_bin=50000,
            random_state=RANDOM_SEED + fold,
            n_jobs=-1,
            verbose=-1
        )
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
        )
    else:
        model = HistGradientBoostingClassifier(
            max_iter=300,
            learning_rate=0.05,
            max_leaf_nodes=31,
            categorical_features='from_dtype',
            random_state=RANDOM_SEED + fold
        )
        model.fit(X_tr, y_tr)
        
    # เก็บค่าความน่าจะเป็นของ Validation และ Test
    oof_probs[val_idx] = model.predict_proba(X_val)
    test_probs += model.predict_proba(X_test) / N_SPLITS
    
    fold_preds = oof_probs[val_idx].argmax(axis=1)
    fold_f1 = f1_score(y_val, fold_preds, average='macro')
    print(f"Fold {fold} | Macro F1: {fold_f1:.5f}")

Model Engine: LightGBM
Fold 1 | Macro F1: 0.74747
Fold 2 | Macro F1: 0.74911
Fold 3 | Macro F1: 0.75168
Fold 4 | Macro F1: 0.75017
Fold 5 | Macro F1: 0.75226


In [14]:
# คำนวณคะแนน Macro F1 รวมทั้งหมด 104,000 แถว
oof_preds_baseline = oof_probs.argmax(axis=1)
baseline_macro_f1 = f1_score(y, oof_preds_baseline, average='macro')

print("=" * 50)
print(f"BASELINE OOF MACRO F1 SCORE: {baseline_macro_f1:.5f}")
print("=" * 50)
print("\nรายละเอียดคะแนนรายคลาส (0=Casual, 1=Grinder, 2=Social, 3=Whale):")
print(classification_report(y, oof_preds_baseline, digits=4))

BASELINE OOF MACRO F1 SCORE: 0.75014

รายละเอียดคะแนนรายคลาส (0=Casual, 1=Grinder, 2=Social, 3=Whale):
              precision    recall  f1-score   support

           0     0.7876    0.8513    0.8182     40957
           1     0.7454    0.7338    0.7395     26032
           2     0.7315    0.6862    0.7081     21029
           3     0.7757    0.6977    0.7346     15982

    accuracy                         0.7649    104000
   macro avg     0.7600    0.7423    0.7501    104000
weighted avg     0.7639    0.7649    0.7634    104000



In [15]:
import shutil

# 1. สร้างไฟล์สำรอง (Backup) ของ submission_final.csv ไว้ก่อนเสมอ เผื่อกรณีฉุกเฉิน
backup_path = SUB_PATH.parent / 'submission_final_backup.csv'
if not backup_path.exists():
    shutil.copy2(SUB_PATH, backup_path)
    print(f"สร้างไฟล์สำรองเรียบร้อยที่: {backup_path.resolve()}")

# 2. โหลดไฟล์ submission_final.csv ล่าสุดจาก Disk (ป้องกันทับงานที่เพื่อนเพิ่งอัปเดต)
sub_latest = pd.read_csv(SUB_PATH)

# เก็บค่าคอลัมน์อื่นทั้งหมดไว้ตรวจสอบหลังอัปเดต
other_cols = ['id', 'task1', 'task3', 'task4', 'task5']
before_other_cols = sub_latest[other_cols].copy()

# 3. สร้างคำทำนายสำหรับ Test Set (25,889 แถว)
test_preds_baseline = test_probs.argmax(axis=1).astype(int)

# 4. ตรวจสอบความถูกต้องก่อนเขียนทับ (Sanity Checks)
assert len(sub_latest) == len(test_preds_baseline), f"จำนวนแถวไม่ตรงกัน! ({len(sub_latest)} vs {len(test_preds_baseline)})"
assert sub_latest['id'].iloc[0] == 'ANS00001' and sub_latest['id'].iloc[-1] == 'ANS25889', "ลำดับ ID ผิดปกติ!"

# 5. อัปเดตเฉพาะคอลัมน์ 'task2' (เติมครบทั้ง 25,889 แถวเป็นจำนวนเต็ม 0, 1, 2, 3)
sub_latest['task2'] = test_preds_baseline

# 6. ตรวจสอบซ้ำหลังอัปเดต (Post-Update Verification)
assert sub_latest['task2'].isnull().sum() == 0, "ยังมีค่า NaN หลงเหลือในคอลัมน์ task2!"
assert set(sub_latest['task2'].unique()).issubset({0, 1, 2, 3}), "พบค่าที่อยู่นอกเหนือคลาส 0, 1, 2, 3!"
assert sub_latest[other_cols].equals(before_other_cols), "อันตราย! ข้อมูลในคอลัมน์อื่นมีการเปลี่ยนแปลง!"

# 7. บันทึกลงไฟล์ submission_final.csv
sub_latest.to_csv(SUB_PATH, index=False)

print("\nบันทึกคำตอบลง 'task2' ใน submission_final.csv สำเร็จ 100%!")
print("\nสัดส่วนคลาสที่ทำนายได้ใน Test Set:")
pred_dist = pd.DataFrame({
    'count': sub_latest['task2'].value_counts().sort_index(),
    'ratio (%)': (sub_latest['task2'].value_counts(normalize=True).sort_index() * 100).round(2)
})
print(pred_dist)

print("\nตัวอย่าง 10 แถวแรกของ submission_final.csv หลังอัปเดต:")
print(sub_latest.head(10))


บันทึกคำตอบลง 'task2' ใน submission_final.csv สำเร็จ 100%!

สัดส่วนคลาสที่ทำนายได้ใน Test Set:
       count  ratio (%)
task2                  
0      11116      42.94
1       6424      24.81
2       4813      18.59
3       3536      13.66

ตัวอย่าง 10 แถวแรกของ submission_final.csv หลังอัปเดต:
         id  task1  task2    task3  task4  task5
0  ANS00001      1      3   370.75      2      1
1  ANS00002      1      2  5036.10      2      0
2  ANS00003      1      2     0.00      3      1
3  ANS00004      1      1  1076.91      0      0
4  ANS00005      0      0  1819.13      3      0
5  ANS00006      1      0   417.28      2      0
6  ANS00007      1      1     0.00      4      0
7  ANS00008      1      0  7080.96      1      0
8  ANS00009      1      3   272.72      1      0
9  ANS00010      1      2   611.45      3      0


## Phase 4: Feature Engineering 9-10

In [27]:
def build_leakfree_features(df_train, df_test):
    tr = df_train.copy()
    te = df_test.copy()
    
    # 1. สร้างคอลัมน์จับคู่ข้ามแกน (Tier x Engagement x VIP)
    for d in [tr, te]:
        d['neg_recency'] = -d['days_since_last_login']
        loy_s = d['loyalty_tier'].astype(str).replace({'nan': 'MISSING', '<NA>': 'MISSING'})
        eng_s = d['engagement_level'].astype(str).replace({'nan': 'MISSING', '<NA>': 'MISSING'})
        vip_s = d['vip_tier'].fillna(-1).astype(int).astype(str)
        
        d['loyalty_x_engagement'] = loy_s + '_' + eng_s
        d['loyalty_x_vip'] = loy_s + '_VIP' + vip_s
        d['engagement_x_vip'] = eng_s + '_VIP' + vip_s

    # 2. สร้าง Full-Axis Latent Scores (คำนวณ Percentile และ Z-Score จาก Train เท่านั้น!)
    spend_axis_cols = [
        'vip_tier', 'spending_frequency', 'avg_monthly_spending', 'rare_items_count',
        'total_spending_thb', 'loyalty_tier_ord', 'item_type_preference_cosmetic',
        'collection_progress', 'achievement_completion_rate'
    ]
    engage_axis_cols = [
        'total_playtime_hours', 'avg_session_duration', 'play_frequency', 'friend_count',
        'tournament_entries', 'engagement_level_ord', 'neg_recency', 'peak_concurrent_hours',
        'speed_of_progression', 'login_streak', 'chat_activity_score', 'win_rate_ranked'
    ]
    
    for axis_name, cols in [('spend', spend_axis_cols), ('engage', engage_axis_cols)]:
        pct_cols, z_cols = [], []
        for col in cols:
            # Percentile Rank จาก Train เท่านั้น
            tr_valid = np.sort(tr[col].dropna().values)
            tr[f'{col}_pct'] = np.where(tr[col].isna(), np.nan, np.searchsorted(tr_valid, tr[col], side='right') / len(tr_valid))
            te[f'{col}_pct'] = np.where(te[col].isna(), np.nan, np.searchsorted(tr_valid, te[col], side='right') / len(tr_valid))
            pct_cols.append(f'{col}_pct')
            
            # Z-Score จาก Train เท่านั้น
            mu, sigma = tr[col].mean(), tr[col].std() + 1e-8
            tr[f'{col}_z'] = (tr[col] - mu) / sigma
            te[f'{col}_z'] = (te[col] - mu) / sigma
            z_cols.append(f'{col}_z')
            
        for d in [tr, te]:
            d[f'{axis_name}_pct_mean'] = d[pct_cols].mean(axis=1)
            d[f'{axis_name}_pct_std'] = d[pct_cols].std(axis=1)
            d[f'{axis_name}_pct_max'] = d[pct_cols].max(axis=1)
            d[f'{axis_name}_pct_min'] = d[pct_cols].min(axis=1)
            
            d[f'{axis_name}_z_mean'] = d[z_cols].mean(axis=1)
            d[f'{axis_name}_z_std'] = d[z_cols].std(axis=1)
            d[f'{axis_name}_z_median'] = d[z_cols].median(axis=1)
            
        tr.drop(columns=pct_cols + z_cols, inplace=True)
        te.drop(columns=pct_cols + z_cols, inplace=True)

    # 3. สร้าง 2x2 Quadrant Interactions & Domain Ratios
    for d in [tr, te]:
        d['whale_pct_prod'] = d['spend_pct_mean'] * d['engage_pct_mean']
        d['whale_pct_harmonic'] = 2 * (d['spend_pct_mean'] * d['engage_pct_mean']) / (d['spend_pct_mean'] + d['engage_pct_mean'] + 1e-5)
        d['social_vs_grinder_pct'] = d['spend_pct_mean'] - d['engage_pct_mean']
        d['total_activity_pct'] = d['spend_pct_mean'] + d['engage_pct_mean']
        
        d['whale_z_sum'] = d['spend_z_mean'] + d['engage_z_mean']
        d['social_vs_grinder_z'] = d['spend_z_mean'] - d['engage_z_mean']
        d['whale_z_prod'] = d['spend_z_mean'] * d['engage_z_mean']
        
        d['spend_per_hour'] = d['total_spending_thb'] / (d['total_playtime_hours'] + 1.0)
        d['monthly_spend_per_freq'] = d['avg_monthly_spending'] / (d['spending_frequency'] + 1.0)
        d['hours_per_freq'] = d['total_playtime_hours'] / (d['play_frequency'] + 1.0)
        d['social_actions_total'] = d['friend_count'] + d['friend_invites_sent'] + d['gifts_sent_received']
        d['cosmetic_vs_perf_pref'] = d['item_type_preference_cosmetic'] - d['item_type_preference_performance']
        d['vip_x_monthly_spend'] = (d['vip_tier'] + 1.0) * d['avg_monthly_spending']
        d['vip_x_spend_freq'] = (d['vip_tier'] + 1.0) * d['spending_frequency']
        d['rare_per_spend'] = d['rare_items_count'] / (d['total_spending_thb'] + 1.0)
        d['vip_to_loyalty_gap'] = d['vip_tier'] - d['loyalty_tier_ord']
        d['ranked_wins_est'] = d['tournament_entries'] * (d['win_rate_ranked'] / 100.0)
        d['chat_per_friend'] = d['chat_activity_score'] / (d['friend_count'] + 1.0)
        
        d['is_loyalty_missing'] = (d['loyalty_tier'].astype(str) == 'MISSING').astype(np.float32)
        d['is_engage_missing'] = (d['engagement_level'].astype(str) == 'MISSING').astype(np.float32)

    # 4. Group Aggregations คำนวณค่าเฉลี่ยจาก Train เท่านั้น แล้ว Map ใส่ Test
    for grp_col in ['loyalty_tier', 'engagement_level', 'loyalty_x_engagement']:
        tr_grp = tr[grp_col].astype(str)
        te_grp = te[grp_col].astype(str)
        for target_num in ['spend_pct_mean', 'engage_pct_mean', 'spend_z_mean', 'engage_z_mean', 'avg_monthly_spending', 'total_playtime_hours']:
            grp_map = tr.groupby(tr_grp)[target_num].mean()
            tr[f'{target_num}_diff_{grp_col}'] = tr[target_num] - tr_grp.map(grp_map).astype(float)
            te[f'{target_num}_diff_{grp_col}'] = te[target_num] - te_grp.map(grp_map).astype(float)

    # 5. ลบฟีเจอร์ Noise และคอลัมน์ชั่วคราว
    noise_cols = [
        'neg_recency', 'account_age_days', 'preferred_game_mode', 'avg_match_length',
        'payment_method', 'region', 'language', 'account_status', 
        'player_type_tag', 'skill_tier', 'skill_tier_ord'
    ]
    tr.drop(columns=[c for c in noise_cols if c in tr.columns], inplace=True)
    te.drop(columns=[c for c in noise_cols if c in te.columns], inplace=True)

    # 6. แปลงคอลัมน์ข้อความทุกชนิดให้ใช้ CategoricalDtype ชุดเดียวกันที่สร้างจาก Train
    cat_cols = [c for c in te.columns if te[c].dtype.name in ['object', 'string', 'str', 'category']]
    for c in cat_cols:
        tr_s = tr[c].astype(str).replace({'nan': 'MISSING', '<NA>': 'MISSING', 'None': 'MISSING'})
        te_s = te[c].astype(str).replace({'nan': 'MISSING', '<NA>': 'MISSING', 'None': 'MISSING'})
        valid_cats = sorted(tr_s.unique().tolist())
        if 'MISSING' not in valid_cats:
            valid_cats.append('MISSING')
        cat_dtype = pd.CategoricalDtype(categories=valid_cats, ordered=False)
        tr[c] = tr_s.astype(cat_dtype)
        te[c] = te_s.where(te_s.isin(valid_cats), 'MISSING').astype(cat_dtype)

    for d in [tr, te]:
        feat_c = [c for c in d.columns if c != TARGET_COL]
        d['missing_features_count'] = d[feat_c].isnull().sum(axis=1)

    y_out = tr[TARGET_COL].values
    X_out = tr.drop(columns=[TARGET_COL])
    return X_out, y_out, te

X_fe, y_fe, X_test_fe = build_leakfree_features(train_clean, test_clean)
print(f"Phase 4 Feature Engineering สำเร็จ (Leak-Free 100%) -> จำนวน Features: {X_fe.shape[1]} คอลัมน์")

Phase 4 Feature Engineering สำเร็จ (Leak-Free 100%) -> จำนวน Features: 90 คอลัมน์


In [28]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)
oof_fe_check = np.zeros((len(X_fe), 4))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_fe, y_fe), 1):
    m_check = HistGradientBoostingClassifier(
        max_iter=500, learning_rate=0.04, max_leaf_nodes=31, min_samples_leaf=40,
        l2_regularization=1.0, categorical_features='from_dtype', random_state=RANDOM_SEED + fold
    )
    m_check.fit(X_fe.iloc[train_idx], y_fe[train_idx])
    oof_fe_check[val_idx] = m_check.predict_proba(X_fe.iloc[val_idx])

fe_only_f1 = f1_score(y_fe, oof_fe_check.argmax(axis=1), average='macro')
print("=" * 55)
print(f"PHASE 4 (FEATURE ENGINEERING ONLY) OOF MACRO F1: {fe_only_f1:.5f}")
print("=" * 55)

PHASE 4 (FEATURE ENGINEERING ONLY) OOF MACRO F1: 0.75175


## Phase 5: Advanced Feature Engineering, Multi-Model Ensembling & Final Safe Submission 12++

In [30]:
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import QuantileTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

# 1. เพิ่ม Smoothed Out-of-Fold Target Encoding (คำนวณแยกในแต่ละ Fold ป้องกัน Data Leakage 100%)
X_mod = X_fe.copy()
X_test_mod = X_test_fe.copy()

te_target_cols = ['loyalty_x_engagement', 'loyalty_x_vip', 'engagement_x_vip']
SMOOTHING = 20.0

for col in te_target_cols:
    for c in range(4):
        X_mod[f'te_{col}_c{c}'] = np.nan
        X_test_mod[f'te_{col}_c{c}'] = 0.0

for fold, (train_idx, val_idx) in enumerate(skf.split(X_mod, y_fe), 1):
    tr_df = X_mod.iloc[train_idx][te_target_cols].copy()
    tr_df['y'] = y_fe[train_idx]
    
    for col in te_target_cols:
        stats = tr_df.groupby(col, observed=False)['y'].agg(['count'])
        for c in range(4):
            prior = (y_fe[train_idx] == c).mean()
            c_sum = tr_df.groupby(col, observed=False)['y'].apply(lambda s: (s == c).sum())
            smooth_prob = (c_sum + SMOOTHING * prior) / (stats['count'] + SMOOTHING)
            
            val_mapped = X_mod.iloc[val_idx][col].map(smooth_prob).astype(float).fillna(prior)
            test_mapped = X_test_mod[col].map(smooth_prob).astype(float).fillna(prior)
            
            X_mod.iloc[val_idx, X_mod.columns.get_loc(f'te_{col}_c{c}')] = val_mapped.values
            X_test_mod[f'te_{col}_c{c}'] += test_mapped.values / N_SPLITS

# 2. เตรียมโครงสร้างข้อมูลสำหรับ 5 โมเดล (ไม่ใช้ pd.concat กับ Test)
model_names = ['lgb', 'xgb', 'cat', 'hgb', 'mlp']
models_oof = {name: np.zeros((len(X_mod), 4)) for name in model_names}
models_test = {name: np.zeros((len(X_test_mod), 4)) for name in model_names}

cat_cols_list = X_mod.select_dtypes(include=['category']).columns.tolist()
X_cat_tr = X_mod.copy()
X_cat_te = X_test_mod.copy()
for c in cat_cols_list:
    X_cat_tr[c] = X_cat_tr[c].astype(str)
    X_cat_te[c] = X_cat_te[c].astype(str)

X_ohe_tr = pd.get_dummies(X_mod, columns=cat_cols_list, dtype=np.float32)
X_ohe_te = pd.get_dummies(X_test_mod, columns=cat_cols_list, dtype=np.float32).reindex(columns=X_ohe_tr.columns, fill_value=0.0)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_mod, y_fe), 1):
    print(f"\n==================== FOLD {fold}/{N_SPLITS} ====================")
    X_tr, y_tr = X_mod.iloc[train_idx], y_fe[train_idx]
    X_val, y_val = X_mod.iloc[val_idx], y_fe[val_idx]
    
    # [1/5] LightGBM (Multi-Depth: num_leaves 31 & 63)
    for cfg in [{'num_leaves': 31, 'col': 0.70, 'seed': 10}, {'num_leaves': 63, 'col': 0.60, 'seed': 20}]:
        m_lgb = lgb.LGBMClassifier(
            objective='multiclass', num_class=4, n_estimators=3500, learning_rate=0.02,
            num_leaves=cfg['num_leaves'], min_child_samples=50, subsample=0.8, subsample_freq=1,
            colsample_bytree=cfg['col'], reg_alpha=1.5, reg_lambda=2.5,
            random_state=RANDOM_SEED + fold + cfg['seed'], n_jobs=-1, verbose=-1
        )
        m_lgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(80, verbose=False)])
        models_oof['lgb'][val_idx] += m_lgb.predict_proba(X_val) / 2.0
        models_test['lgb'] += m_lgb.predict_proba(X_test_mod) / (N_SPLITS * 2.0)
    print(f"  [1/5] LightGBM (Multi-Depth) F1: {f1_score(y_val, models_oof['lgb'][val_idx].argmax(axis=1), average='macro'):.5f}")

    # [2/5] XGBoost (Multi-Depth: max_depth 5 & 7)
    for cfg in [{'depth': 5, 'col': 0.70, 'seed': 30}, {'depth': 7, 'col': 0.60, 'seed': 40}]:
        m_xgb = xgb.XGBClassifier(
            objective='multi:softprob', num_class=4, n_estimators=3500, learning_rate=0.02,
            max_depth=cfg['depth'], min_child_weight=25, subsample=0.8,
            colsample_bytree=cfg['col'], reg_alpha=1.5, reg_lambda=2.5,
            enable_categorical=True, tree_method='hist', early_stopping_rounds=80,
            random_state=RANDOM_SEED + fold + cfg['seed'], n_jobs=-1
        )
        m_xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        models_oof['xgb'][val_idx] += m_xgb.predict_proba(X_val) / 2.0
        models_test['xgb'] += m_xgb.predict_proba(X_test_mod) / (N_SPLITS * 2.0)
    print(f"  [2/5] XGBoost  (Multi-Depth) F1: {f1_score(y_val, models_oof['xgb'][val_idx].argmax(axis=1), average='macro'):.5f}")

    # [3/5] CatBoost
    m_cat = CatBoostClassifier(
        loss_function='MultiClass', iterations=3500, learning_rate=0.035,
        depth=6, l2_leaf_reg=6.0, cat_features=cat_cols_list,
        random_seed=RANDOM_SEED + fold, early_stopping_rounds=100, verbose=0
    )
    m_cat.fit(X_cat_tr.iloc[train_idx], y_tr, eval_set=(X_cat_tr.iloc[val_idx], y_val))
    models_oof['cat'][val_idx] = m_cat.predict_proba(X_cat_tr.iloc[val_idx])
    models_test['cat'] += m_cat.predict_proba(X_cat_te) / N_SPLITS
    print(f"  [3/5] CatBoost               F1: {f1_score(y_val, models_oof['cat'][val_idx].argmax(axis=1), average='macro'):.5f}")

    # [4/5] HistGradientBoosting
    m_hgb = HistGradientBoostingClassifier(
        max_iter=1000, learning_rate=0.025, max_leaf_nodes=45, min_samples_leaf=50,
        l2_regularization=2.5, categorical_features='from_dtype', random_state=RANDOM_SEED + fold
    )
    m_hgb.fit(X_tr, y_tr)
    models_oof['hgb'][val_idx] = m_hgb.predict_proba(X_val)
    models_test['hgb'] += m_hgb.predict_proba(X_test_mod) / N_SPLITS
    print(f"  [4/5] HistGradientBoosting   F1: {f1_score(y_val, models_oof['hgb'][val_idx].argmax(axis=1), average='macro'):.5f}")

    # [5/5] Deep MLP Neural Network (2 Seeds, Fit Scaler เฉพาะใน Train Fold)
    for mlp_seed in [50, 60]:
        m_mlp = make_pipeline(
            SimpleImputer(strategy='median'),
            QuantileTransformer(n_quantiles=1500, output_distribution='normal', random_state=RANDOM_SEED + mlp_seed),
            MLPClassifier(
                hidden_layer_sizes=(256, 128, 64), activation='relu', alpha=0.12,
                batch_size=512, learning_rate_init=0.0008, max_iter=60,
                early_stopping=True, validation_fraction=0.1, random_state=RANDOM_SEED + fold + mlp_seed
            )
        )
        m_mlp.fit(X_ohe_tr.iloc[train_idx], y_tr)
        models_oof['mlp'][val_idx] += m_mlp.predict_proba(X_ohe_tr.iloc[val_idx]) / 2.0
        models_test['mlp'] += m_mlp.predict_proba(X_ohe_te) / (N_SPLITS * 2.0)
    print(f"  [5/5] Deep MLP (2-Seed)      F1: {f1_score(y_val, models_oof['mlp'][val_idx].argmax(axis=1), average='macro'):.5f}")

print("\n================ สรุปคะแนน OOF รายโมเดล ================")
for name in model_names:
    sc = f1_score(y_fe, models_oof[name].argmax(axis=1), average='macro')
    print(f"  {name.upper():<5} OOF Macro F1: {sc:.5f}")


==================== FOLD 1/5 ====================
  [1/5] LightGBM (Multi-Depth) F1: 0.75185
  [2/5] XGBoost  (Multi-Depth) F1: 0.75340
  [3/5] CatBoost               F1: 0.75202
  [4/5] HistGradientBoosting   F1: 0.74997
  [5/5] Deep MLP (2-Seed)      F1: 0.74767

==================== FOLD 2/5 ====================
  [1/5] LightGBM (Multi-Depth) F1: 0.75013
  [2/5] XGBoost  (Multi-Depth) F1: 0.75123
  [3/5] CatBoost               F1: 0.75190
  [4/5] HistGradientBoosting   F1: 0.74870
  [5/5] Deep MLP (2-Seed)      F1: 0.75102

==================== FOLD 3/5 ====================
  [1/5] LightGBM (Multi-Depth) F1: 0.75760
  [2/5] XGBoost  (Multi-Depth) F1: 0.75645
  [3/5] CatBoost               F1: 0.75863
  [4/5] HistGradientBoosting   F1: 0.75301
  [5/5] Deep MLP (2-Seed)      F1: 0.75039

==================== FOLD 4/5 ====================
  [1/5] LightGBM (Multi-Depth) F1: 0.75538
  [2/5] XGBoost  (Multi-Depth) F1: 0.75482
  [3/5] CatBoost               F1: 0.75413
  [4/5] HistGradie

In [31]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from scipy.optimize import minimize

def build_meta_features(probs_dict, names):
    prob_list = [probs_dict[m] for m in names]
    probs_concat = np.hstack(prob_list)
    eps = 1e-6
    clipped = np.clip(probs_concat, eps, 1.0 - eps)
    log_odds = np.log(clipped)
    stacked_3d = np.stack(prob_list, axis=0)
    mean_probs = stacked_3d.mean(axis=0)
    std_probs = stacked_3d.std(axis=0)
    entropy = -np.sum(np.clip(mean_probs, eps, 1.0) * np.log(np.clip(mean_probs, eps, 1.0)), axis=1, keepdims=True)
    return np.hstack([probs_concat, log_odds, std_probs, entropy])

meta_X_oof = build_meta_features(models_oof, model_names)
meta_X_test = build_meta_features(models_test, model_names)

oof_meta_probs = np.zeros((len(y_fe), 4))
test_meta_probs = np.zeros((len(X_test_mod), 4))

for fold, (train_idx, val_idx) in enumerate(skf.split(meta_X_oof, y_fe), 1):
    meta_lr = make_pipeline(StandardScaler(), LogisticRegression(C=0.2, max_iter=1000, random_state=RANDOM_SEED))
    meta_lr.fit(meta_X_oof[train_idx], y_fe[train_idx])
    
    meta_mlp = make_pipeline(
        StandardScaler(),
        MLPClassifier(hidden_layer_sizes=(32, 16), alpha=1.0, max_iter=40, early_stopping=True, random_state=RANDOM_SEED + fold)
    )
    meta_mlp.fit(meta_X_oof[train_idx], y_fe[train_idx])
    
    oof_meta_probs[val_idx] = 0.6 * meta_lr.predict_proba(meta_X_oof[val_idx]) + 0.4 * meta_mlp.predict_proba(meta_X_oof[val_idx])
    test_meta_probs += (0.6 * meta_lr.predict_proba(meta_X_test) + 0.4 * meta_mlp.predict_proba(meta_X_test)) / N_SPLITS

print(f"PHASE 5 STACKING OOF MACRO F1 (RAW): {f1_score(y_fe, oof_meta_probs.argmax(axis=1), average='macro'):.5f}")

# หา Class Multipliers [1.0, w1, w2, w3] จาก Train/Val Split เท่านั้น (ถูกกติกา 100%)
def optimize_3d_multipliers(probs, y_true):
    best_w = np.ones(4)
    best_score = f1_score(y_true, probs.argmax(axis=1), average='macro')
    for grid in [np.linspace(0.75, 1.35, 61), np.linspace(0.92, 1.12, 41)]:
        for _ in range(3):
            for c in [1, 2, 3]:
                best_c_w = best_w[c]
                for w_cand in grid:
                    w_try = best_w.copy()
                    w_try[c] = w_cand
                    sc = f1_score(y_true, (probs * w_try).argmax(axis=1), average='macro')
                    if sc > best_score:
                        best_score = sc
                        best_c_w = w_cand
                best_w[c] = best_c_w
    res = minimize(
        lambda w3: -f1_score(y_true, (probs * np.array([1.0, w3[0], w3[1], w3[2]])).argmax(axis=1), average='macro'),
        best_w[1:], method='Powell', options={'maxiter': 1000, 'ftol': 1e-6}
    )
    if -res.fun > best_score:
        best_w = np.array([1.0, res.x[0], res.x[1], res.x[2]])
    return best_w

oof_final_preds = np.zeros(len(y_fe), dtype=int)
fold_mults = []

for fold, (train_idx, val_idx) in enumerate(skf.split(meta_X_oof, y_fe), 1):
    w_fold = optimize_3d_multipliers(oof_meta_probs[train_idx], y_fe[train_idx])
    fold_mults.append(w_fold)
    oof_final_preds[val_idx] = (oof_meta_probs[val_idx] * w_fold).argmax(axis=1)

final_mult = np.mean(fold_mults, axis=0)
final_score = f1_score(y_fe, oof_final_preds, average='macro')

print("Optimal Fixed Multipliers (from Train/Val only):", np.round(final_mult, 4))
print("=" * 65)
print(f"FINAL PHASE 5 LEAK-FREE OOF MACRO F1: {final_score:.5f}")
print("=" * 65)
print(classification_report(y_fe, oof_final_preds, digits=4))

# นำ Multipliers คงที่ไปใช้กับ Test Set และอัปเดต submission_final.csv อย่างปลอดภัย
test_final_preds = (test_meta_probs * final_mult).argmax(axis=1).astype(int)

sub_latest = pd.read_csv(SUB_PATH)
other_cols = ['id', 'task1', 'task3', 'task4', 'task5']
before_other_cols = sub_latest[other_cols].copy()

assert len(sub_latest) == len(test_final_preds), "จำนวนแถวไม่ตรงกัน!"
assert sub_latest['id'].iloc[0] == 'ANS00001' and sub_latest['id'].iloc[-1] == 'ANS25889', "ลำดับ ID ผิดปกติ!"

sub_latest['task2'] = test_final_preds

assert sub_latest['task2'].isnull().sum() == 0, "ยังมีค่า NaN หลงเหลือ!"
assert set(sub_latest['task2'].unique()).issubset({0, 1, 2, 3}), "พบค่าคลาสนอกเหนือจาก 0, 1, 2, 3!"
assert sub_latest[other_cols].equals(before_other_cols), "อันตราย! คอลัมน์อื่นถูกเปลี่ยนแปลง!"

sub_latest.to_csv(SUB_PATH, index=False)
print(f"\nบันทึกผลลัพธ์ชุด Leak-Free 100% ลง '{SUB_PATH.resolve()}' เรียบร้อย!")
print("สัดส่วนคลาสที่ทำนายได้ใน Test Set:")
print(pd.Series(test_final_preds).value_counts(normalize=True).sort_index().round(4) * 100)

PHASE 5 STACKING OOF MACRO F1 (RAW): 0.75539
Optimal Fixed Multipliers (from Train/Val only): [1.     1.0585 1.0745 1.1612]
FINAL PHASE 5 LEAK-FREE OOF MACRO F1: 0.75525
              precision    recall  f1-score   support

           0     0.7968    0.8481    0.8216     40957
           1     0.7446    0.7446    0.7446     26032
           2     0.7395    0.6826    0.7099     21029
           3     0.7703    0.7211    0.7449     15982

    accuracy                         0.7692    104000
   macro avg     0.7628    0.7491    0.7553    104000
weighted avg     0.7681    0.7692    0.7680    104000


บันทึกผลลัพธ์ชุด Leak-Free 100% ลง 'C:\Users\panuw\CPE342-Hackathon\submission_final.csv' เรียบร้อย!
สัดส่วนคลาสที่ทำนายได้ใน Test Set:
0    42.23
1    24.81
2    18.44
3    14.52
Name: proportion, dtype: float64


In [32]:
from pathlib import Path
import numpy as np
import pandas as pd

# 1. หาตำแหน่งไฟล์ submission_final.csv
SUB_PATH = Path('../submission_final.csv') if Path('train.csv').exists() else Path('submission_final.csv')
BACKUP_PATH = SUB_PATH.parent / 'submission_final_backup.csv'

sub_current = pd.read_csv(SUB_PATH)

# 2. กู้คืน 4 แถวตัวอย่างแรก (ANS00001 - ANS00004) และคงค่าทำนาย Phase 5 ตั้งแต่ ANS00005 เป็นต้นไป
if BACKUP_PATH.exists():
    sub_orig = pd.read_csv(BACKUP_PATH)
    other_cols = ['id', 'task1', 'task3', 'task4', 'task5']
    assert sub_current[other_cols].equals(sub_orig[other_cols]), "คอลัมน์อื่นไม่ตรงกับไฟล์สำรอง!"
    
    # เติมเฉพาะช่องที่เคยเป็น NaN ในไฟล์ต้นฉบับ (ตั้งแต่ index 4 / ANS00005 เป็นต้นไป)
    empty_mask = sub_orig['task2'].isna()
    fixed_task2 = sub_orig['task2'].copy()
    fixed_task2.loc[empty_mask] = sub_current.loc[empty_mask, 'task2'].astype('float64')
    sub_current['task2'] = fixed_task2.astype('float64')
else:
    sub_current['task2'] = sub_current['task2'].astype('float64')
    sub_current.loc[0:3, 'task2'] = [2.0, 1.0, 3.0, 2.0]

# 3. ตรวจสอบความถูกต้องอย่างเข้มงวดก่อนเซฟ
assert sub_current['task2'].dtype == np.float64, "ชนิดข้อมูลต้องเป็น float64 เพื่อให้มี .0 ลงท้าย"
assert sub_current.loc[0:3, 'task2'].tolist() == [2.0, 1.0, 3.0, 2.0], "4 แถวตัวอย่างแรกไม่ตรงกับต้นฉบับ!"
assert sub_current['task2'].isna().sum() == 0, "ยังมีค่าว่างหลงเหลือใน task2!"
assert set(sub_current['task2'].unique()).issubset({0.0, 1.0, 2.0, 3.0}), "พบค่าที่อยู่นอกเหนือ 0.0, 1.0, 2.0, 3.0!"

# 4. บันทึกทับลง submission_final.csv
sub_current.to_csv(SUB_PATH, index=False)

print("แก้ไขเรียบร้อย! ตัวอย่าง 10 บรรทัดแรกของไฟล์ submission_final.csv จริง:")
with open(SUB_PATH, 'r') as f:
    for _ in range(10):
        print(f.readline().strip())

แก้ไขเรียบร้อย! ตัวอย่าง 10 บรรทัดแรกของไฟล์ submission_final.csv จริง:
id,task1,task2,task3,task4,task5
ANS00001,1,2.0,370.75,2,1
ANS00002,1,1.0,5036.1,2,0
ANS00003,1,3.0,0.0,3,1
ANS00004,1,2.0,1076.91,0,0
ANS00005,0,0.0,1819.13,3,0
ANS00006,1,0.0,417.28,2,0
ANS00007,1,1.0,0.0,4,0
ANS00008,1,1.0,7080.96,1,0
ANS00009,1,3.0,272.72,1,0
